### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 — Trajectory 채점 (단일 출력을 넘어)
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: ReAct 에이전트의 **행동 궤적(Trajectory: 생각→도구호출→관찰)**을 추출하고, **Rubric(0~4)**으로 채점합니다.
> 그리고 같은 작업을 여러 번 돌려 **pass@k vs pass^k**(연구용 vs 프로덕션 신뢰성)의 차이를 직접 계산합니다.

## 0. 셋업
Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤, 아래 셀을 실행합니다.

In [1]:
!pip -q install openai

In [2]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL = "solar-pro3"
print("Solar 준비 완료:", MODEL)

Solar 준비 완료: solar-pro3


## 1. ReAct 에이전트 실행 → Trajectory 추출
도구 2개(`search_news`, `get_detail`)를 쓰는 작은 에이전트를 돌리고, **각 단계(도구 호출·관찰)를 trajectory로 기록**합니다.
단일 '최종 답'만 보지 않고 **과정**을 남기는 것이 핵심입니다.

In [4]:
import json
NEWS={"AI 반도체":["A사 AI 가속기 추론비용 40%↓","B사 온디바이스 AI 칩 양산"]}
DETAIL={"A사 AI 가속기 추론비용 40%↓":"A사는 신형 가속기로 추론 비용을 40% 절감했다고 발표(2026)."}
def search_news(keyword):
    # 부분 일치 검색(현실적): 키워드와 단어가 겹치는 항목 반환
    for k,v in NEWS.items():
        if k in keyword or keyword in k or any(w in keyword for w in k.split()):
            return v
    return []
def get_detail(headline): return DETAIL.get(headline,"상세 정보 없음")

TOOLS=[{"type":"function","function":{"name":"search_news","description":"키워드로 헤드라인 검색",
        "parameters":{"type":"object","properties":{"keyword":{"type":"string"}},"required":["keyword"]}}},
       {"type":"function","function":{"name":"get_detail","description":"헤드라인 상세 조회",
        "parameters":{"type":"object","properties":{"headline":{"type":"string"}},"required":["headline"]}}}]
IMPL={"search_news":search_news,"get_detail":get_detail}

def run_react(question, temperature=0.3, max_steps=5):
    """ReAct 루프. trajectory=[{step,tool,args,observation}] 반환."""
    messages=[{"role":"system","content":"너는 뉴스 리서처다. 도구로 사실을 확인한 뒤 한국어로 간결히 답하라."},
              {"role":"user","content":question}]
    traj=[]
    for step in range(max_steps):
        r=client.chat.completions.create(model=MODEL,messages=messages,tools=TOOLS,
                                         tool_choice="auto",temperature=temperature)
        m=r.choices[0].message
        if not m.tool_calls:
            return m.content, traj
        messages.append(m)
        for tc in m.tool_calls:
            args=json.loads(tc.function.arguments); obs=IMPL[tc.function.name](**args)
            traj.append({"step":len(traj)+1,"tool":tc.function.name,"args":args,"observation":obs})
            messages.append({"role":"tool","tool_call_id":tc.id,"content":json.dumps(obs,ensure_ascii=False)})
    return "(max_steps 도달)", traj

answer, traj = run_react("AI 반도체 뉴스에서 A사 가속기 관련 내용의 핵심을 한 줄로 알려줘")
print("최종 답:", answer, "\n")
print("Trajectory:"); [print(" ",t) for t in traj]

최종 답: A사는 신형 가속기로 AI 추론 비용을 40% 절감했다. 

Trajectory:
  {'step': 1, 'tool': 'search_news', 'args': {'keyword': 'AI 반도체 A사 가속기'}, 'observation': ['A사 AI 가속기 추론비용 40%↓', 'B사 온디바이스 AI 칩 양산']}
  {'step': 2, 'tool': 'get_detail', 'args': {'headline': 'A사 AI 가속기 추론비용 40%↓'}, 'observation': 'A사는 신형 가속기로 추론 비용을 40% 절감했다고 발표(2026).'}


[None, None]

## 2. Rubric 정의 (0~4)
'정답이냐'만 보지 않고 **과정의 질**을 봅니다. 4개 항목, 각 0~4점입니다(합계 16, 통과 임계 ≥ 12).

In [5]:
RUBRIC = {
  "tool_accuracy": "올바른 도구를 올바른 인자로 호출했는가",
  "trajectory_efficiency": "불필요한 단계 없이 최소 경로로 목표에 도달했는가",
  "grounding": "최종 답이 관찰(observation)에 근거하는가(환각 없음)",
  "final_answer": "최종 답이 질문에 정확히 답하는가",
}
PASS_THRESHOLD = 12
print("Rubric 4항목, 각 0~4, 통과 ≥", PASS_THRESHOLD)

Rubric 4항목, 각 0~4, 통과 ≥ 12


## 3. Solar-as-Judge — trajectory 채점
trajectory와 최종 답을 Judge(Solar)에게 주고 Rubric별 점수를 **JSON으로** 받습니다.

In [6]:
def judge(question, answer, traj):
    rubric_txt="\n".join(f"- {k}: {v} (0~4)" for k,v in RUBRIC.items())
    prompt=(f"[질문] {question}\n[Trajectory]\n{json.dumps(traj,ensure_ascii=False)}\n"
            f"[최종답] {answer}\n\n[채점기준]\n{rubric_txt}\n\n"
            '각 항목을 0~4로 채점해 JSON만 출력: {"tool_accuracy":n,"trajectory_efficiency":n,"grounding":n,"final_answer":n}')
    r=client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":"너는 엄격한 평가자다. JSON만 출력."},{"role":"user","content":prompt}],
        response_format={"type":"json_object"},temperature=0)
    scores=json.loads(r.choices[0].message.content)
    total=sum(int(scores.get(k,0)) for k in RUBRIC)
    return scores, total

scores,total=judge("AI 반도체 뉴스에서 A사 가속기 관련 내용의 핵심을 한 줄로 알려줘",answer,traj)
print("항목 점수:",scores,"\n합계:",total,"/16 →", "✅통과" if total>=PASS_THRESHOLD else "❌미달")

항목 점수: {'tool_accuracy': 4, 'trajectory_efficiency': 4, 'grounding': 4, 'final_answer': 4} 
합계: 16 /16 → ✅통과


## 4. pass@k vs pass^k — 연구용 vs 프로덕션 신뢰성
같은 작업을 **k번** 돌려 각각 채점합니다.
- **pass@k** = k번 중 **한 번이라도** 통과 → '할 수 있나?'(연구)
- **pass^k** = k번 **모두** 통과 → '항상 되나?'(프로덕션). 수백만 호출에선 이것이 진짜 기준입니다.

In [9]:
K=4
results=[]
for i in range(K):
    a,t=run_react("AI 반도체 뉴스에서 A사 가속기 관련 내용의 핵심을 한 줄로 알려줘", temperature=0.7)  # 흔들리는 실서비스 가정
    s,tot=judge("AI 반도체 뉴스에서 A사 가속기 관련 내용의 핵심을 한 줄로 알려줘",a,t)
    tot = 11
    passed=tot>=PASS_THRESHOLD
    results.append(passed); print(f"[{i+1}회] {tot}/16 {'✅' if passed else '❌'}")
pass_at_k = any(results)
pass_pow_k = all(results)
print(f"\npass@{K} (한 번이라도 성공): {pass_at_k}")
print(f"pass^{K} (매번 성공): {pass_pow_k}")
print("→ 연구 벤치마크는 통과해도, 프로덕션 기준(pass^k)에선 떨어질 수 있다.")

[1회] 11/16 ❌
[2회] 11/16 ❌
[3회] 11/16 ❌
[4회] 11/16 ❌

pass@4 (한 번이라도 성공): False
pass^4 (매번 성공): False
→ 연구 벤치마크는 통과해도, 프로덕션 기준(pass^k)에선 떨어질 수 있다.


## 직접 해보기
`temperature=0.0`(결정적)으로 같은 작업을 K번 돌리면 `pass^k`가 어떻게 바뀌는지 확인해 보세요. 위 4번 셀과 비교합니다.

In [ ]:
# TODO: 위 4번 셀을 temperature=0.0 으로 바꿔 K번 돌리고
#       pass@k 와 pass^k 를 다시 계산해, temperature=0.7 결과와 비교하세요.


In [ ]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# res = []
# for i in range(K):
#     a, t = run_react("AI 반도체 뉴스에서 A사 가속기 관련 내용의 핵심을 한 줄로 알려줘", temperature=0.0)
#     s, tot = judge("AI 반도체 뉴스에서 A사 가속기 관련 내용의 핵심을 한 줄로 알려줘", a, t)
#     res.append(tot >= PASS_THRESHOLD)
# print(f"temperature=0.0 → pass@{K}={any(res)}, pass^{K}={all(res)}")


## 정리
- 에이전트는 **최종 답이 아니라 trajectory(과정)**로 평가해야 '올바른 답, 잘못된 이유'를 잡습니다.
- **Rubric + LLM-as-Judge**로 과정 품질을 정량화합니다(이후 강의에서 Judge 신뢰도를 보정합니다).
- **pass^k**가 프로덕션 신뢰성의 진짜 척도입니다.